# 02 Integración

**Resultado:** tablas listas para analizar en 02_datos/02_Analitico

**Índice**

1. Cargar
2. Unir
3. Variables de fecha
4. Variables de negocio
5. Tablas por grano
6. Guardar

## 0. Librerías y rutas

In [55]:
# Librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from dotenv import load_dotenv

# Opciones de vista
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Rutas del proyecto (el notebook vive en 03_notebooks)
RAIZ = Path.cwd().parent
ORIGINALES = RAIZ / '02_datos' / '01_Originales'
ANALITICO = RAIZ / '02_datos' / '02_Analitico'
RESULTADOS = RAIZ / '06_resultados'

# Variables de entorno (.env): ahí va DATABASE_URL
load_dotenv(RAIZ / '.env')

False

## 1. Cargar

Los datos limpios del notebook 01.

In [56]:
p1g = pd.read_pickle(ANALITICO / 'Planta1_Generacion.pickle')
p2g = pd.read_pickle(ANALITICO / 'Planta2_Generacion.pickle')
p1s = pd.read_pickle(ANALITICO / 'Planta1_Sensores.pickle')
p2s = pd.read_pickle(ANALITICO / 'Planta2_Sensores.pickle')

## 2. Unir

Juntar las fuentes. El total antes y después tiene que cuadrar.

In [57]:
#concatenaciones de los dataframes
gener = pd.concat([p1g, p2g], ignore_index=True)
sensores = pd.concat([p1s, p2s], ignore_index=True)

#unión de todos los dataframes por columna id_planta (join), no usamos inner porque queremos conservar todos
# los registros de generacion y agregar los de sensores que coincidan

df = gener.merge(sensores, on=['fecha_hora', 'id_planta'], how='left')

## 3. Variables de fecha

Año, mes, día de la semana, hora, festivos.

In [58]:
df['mes'] = df['fecha_hora'].dt.month
df['dia'] = df['fecha_hora'].dt.day
df['hora'] = df['fecha_hora'].dt.hour
df['minuto'] = df['fecha_hora'].dt.minute
df['segundo'] = df['fecha_hora'].dt.second

df = df.set_index('fecha_hora')

## 4. Variables de negocio

Eficiencia de conversión del inversor, AC sobre DC en porcentaje y se ordena

In [59]:
def eficiencia_interverter(dc,ac):
    # Si DC es 0 por noche u otra razon, es 100% la eficiencia, convertirlo a 0 distorsiona el analisis
    temp = np.where(dc>0,ac/dc,1) * 100
    return np.where(temp>100,100,temp)


### Insight 4

**Sin sol no hay eficiencia que medir**. Con DC en cero, un 0 marcaría una avería inexistente en cada hora nocturna, así que se deja en 100 para no ensuciar la serie. Ese 100 es relleno, no medición: las medias y rankings deben excluir las horas de noche con una bandera, que aún está pendiente

In [60]:
df['eficiencia'] = eficiencia_interverter(df['potencia_dc_kw'],df['potencia_ac_kw'])

In [61]:
#Ordenar las variables
orden = ['id_planta','id_inversor','id_sensor_meteorologico','mes','dia','hora','minuto',
         'irradiacion_wh_m2','temperatura_ambiente_c','temperatura_modulo_c',
         'potencia_dc_kw','eficiencia','potencia_ac_kw','energia_diaria_kwh','energia_total_kwh']
df = df[orden]

## 5. Grano y tablas

Segun las entidades de tabla, vamos a hacer dos agregacion
- Inverter y Planta por Dia

### Tabla Inverter - Planta por Dia

In [64]:
# algunos como las temperaturas no se agregan por suma al no tener sentido y la energia_diaria_kwh
# ya es un acumulado por inversor, agrupamos por planta e inversor y calculamos estadísticas por día
# indice es tipo fecha para que resample funcione

resumen = df.groupby(['id_planta', 'id_inversor']).resample('D').agg({
    'irradiacion_wh_m2': ['min', 'mean', 'max', 'sum'],
    'temperatura_ambiente_c': ['min', 'mean', 'max'],
    'temperatura_modulo_c': ['min', 'mean', 'max'],
    'potencia_dc_kw': ['min', 'mean', 'max'],
    'eficiencia': ['min', 'mean', 'max'],
    'potencia_ac_kw': ['min', 'mean', 'max'],
    'energia_diaria_kwh': ['min', 'mean', 'max']
})

# convertimos las columnas multinivel en nombres simples, como "eficiencia_mean"
columnas_planas = []
for variable, estadistico in resumen.columns:
    columnas_planas.append(f'{variable}_{estadistico}')

resumen.columns = columnas_planas

# pasamos planta, inversor y fecha del índice a columnas normales
resumen = resumen.reset_index()

#volvemos a hacer fecha_hora como index.
df_diario = resumen.copy()
df_diario = resumen.set_index('fecha_hora')

## Conclusión

Se integraron las fuentes con una unión *left* por fecha y planta y se generó un resumen diario por planta e inversor de **8.140 filas y 24 columnas**. Las métricas de generación tienen datos en **1.464 filas** y las de sensores en **1.222**; parte de las filas regularizadas no tiene lecturas. Aunque la suma de irradiación aparece completa, sus ceros **no deben interpretarse automáticamente como mediciones.**

## 6. Guardar

Cada tabla a 02_datos/02_Analitico.

In [68]:
df.to_pickle(ANALITICO / 'tablon_analitico.pkl')
df_diario.to_pickle(ANALITICO/'tablon_analitico_diario.pkl')